In [23]:
# ======================================
# 1. CONFIGURAÇÕES E PARÂMETROS GERAIS
# ======================================

import pandas as pd
import numpy as np
import random

qtd_fornecedores = 200
meses = 24
tx_anomalia_mes = 0.06
seed = 42

random.seed(seed)
np.random.seed(seed)

In [24]:
# ======================================
# 2. GERAÇÃO DE FORNECEDORES
# ======================================

gp_crescente = ['crescente'] * 80
gp_decrescente = ['decrescente'] * 60
gp_estavel = ['estavel'] * 60

grupos = gp_crescente + gp_decrescente + gp_estavel
random.shuffle(grupos)

fornecedores = []

for i in range(200):
    opcao = grupos[i]
    padroes = ['NUM', 'ACC', 'FORN']

    padrao_atual = random.choice(padroes)

    if padrao_atual == 'NUM':
        id_fornecedor = f'{1000000+i}'
    
    elif padrao_atual == 'ACC':
        id_fornecedor = f'ACC-{i+1:05d}'

    elif padrao_atual == 'FORN':
        id_fornecedor = f'FORN-{i+1:05d}'
    
    valor_medio_inicial = round(np.random.lognormal(8, 0.6), 2)

    if valor_medio_inicial <= 4000:
        freq_media_inicial = np.random.randint(10, 19)
    elif valor_medio_inicial > 4000 and valor_medio_inicial <= 10000:
        freq_media_inicial = np.random.randint(6, 13)
    elif valor_medio_inicial > 10000:
        freq_media_inicial = np.random.randint(3, 9)

    prob_inicio = np.random.uniform(0, 1)

    if prob_inicio < 0.6:
        mes_inicio = np.random.randint(1, 4)
    else:
        mes_inicio = np.random.randint(4, 24)

    if mes_inicio > 18:
        mes_fim = 24
    else:
        prob_fim = np.random.uniform(0, 1)
        if prob_fim < 0.6:
            mes_fim = 24
        else:
            mes_fim = np.random.randint(mes_inicio + 6, 25)

    fornecedores.append({
        'id_fornecedor_raw': id_fornecedor,
        'grupo': opcao,
        'valor_medio_inicial': valor_medio_inicial,
        'freq_media_inicial': freq_media_inicial,
        'mes_inicio': mes_inicio,
        'mes_fim': mes_fim
    })
df_fornecedores = pd.DataFrame(fornecedores)

# print(mes_inicio, mes_fim)

#print(df_fornecedores)

In [25]:
# ======================================
# 3. ESTRUTURA TEMPORAL (FORNECEDOR x MÊS)
# ======================================

df_meses = pd.DataFrame({'mes_pagamento': pd.RangeIndex(start=1, stop=25, step=1)})

df_fornecedor_mes_completo = df_fornecedores.merge(df_meses, how='cross')

df_fornecedor_mes_ativo = df_fornecedor_mes_completo.loc[
    (df_fornecedor_mes_completo['mes_pagamento']>=df_fornecedor_mes_completo['mes_inicio']) & 
    (df_fornecedor_mes_completo['mes_pagamento']<=df_fornecedor_mes_completo['mes_fim'])
]

display(df_fornecedor_mes_ativo)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento
9,1000000,crescente,4015.95,8,10,24,10
10,1000000,crescente,4015.95,8,10,24,11
11,1000000,crescente,4015.95,8,10,24,12
12,1000000,crescente,4015.95,8,10,24,13
13,1000000,crescente,4015.95,8,10,24,14
...,...,...,...,...,...,...,...
4790,1000199,estavel,2483.11,13,3,19,15
4791,1000199,estavel,2483.11,13,3,19,16
4792,1000199,estavel,2483.11,13,3,19,17
4793,1000199,estavel,2483.11,13,3,19,18


In [26]:
# ======================================
# 4. MODELAGEM DE FREQUÊNCIA (POISSON)
# ======================================

taxa_freq = 0.015

min_lambda = 1.0
df_fornecedor_mes_ativo['mes_relativo'] = df_fornecedor_mes_ativo['mes_pagamento'] - df_fornecedor_mes_ativo['mes_inicio']
df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo['freq_media_inicial']
#print(df_fornecedor_mes_ativo)

mask_crescente = df_fornecedor_mes_ativo['grupo'] == 'crescente'
#display(df_fornecedor_mes_ativo)
#display(mask_crescente)
df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo.lambda_mes.astype(float)

# Crescente
df_fornecedor_mes_ativo.loc[ 
    mask_crescente, 
    'lambda_mes' 
    ] = df_fornecedor_mes_ativo['freq_media_inicial'] * (1 + taxa_freq) ** df_fornecedor_mes_ativo['mes_relativo']


mask_decrescente = df_fornecedor_mes_ativo['grupo'] == 'decrescente'
# Decrescente
df_fornecedor_mes_ativo.loc[ 
    mask_decrescente, 
    'lambda_mes' 
    ] = (df_fornecedor_mes_ativo['freq_media_inicial'] * (1 - taxa_freq) ** df_fornecedor_mes_ativo['mes_relativo']).clip(lower=min_lambda)

# display(df_fornecedor_mes_ativo[df_fornecedor_mes_ativo['id_fornecedor_raw'] == 'FORN-00003'])



In [27]:
# ======================================
# 5. MODELAGEM DE VALOR MÉDIO MENSAL
# ======================================

taxa_valor = 0.005

df_fornecedor_mes_ativo["quantidade_real_pagamentos"] = np.random.poisson(df_fornecedor_mes_ativo["lambda_mes"])

# Crescente
df_fornecedor_mes_ativo.loc[ 
    mask_crescente, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial'] * (1 + taxa_valor) ** df_fornecedor_mes_ativo['mes_relativo']

# Decrescente
df_fornecedor_mes_ativo.loc[ 
    mask_decrescente, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial'] * (1 - taxa_valor) ** df_fornecedor_mes_ativo['mes_relativo']

# Estável
mask_estavel = df_fornecedor_mes_ativo['grupo'] == 'estavel'
df_fornecedor_mes_ativo.loc[ 
    mask_estavel, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial']

display(df_fornecedor_mes_ativo)
#display(df_fornecedor_mes_ativo)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes
9,1000000,crescente,4015.95,8,10,24,10,0,8.000000,7,4015.950000
10,1000000,crescente,4015.95,8,10,24,11,1,8.120000,8,4036.029750
11,1000000,crescente,4015.95,8,10,24,12,2,8.241800,13,4056.209899
12,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
13,1000000,crescente,4015.95,8,10,24,14,4,8.490908,12,4096.873403
...,...,...,...,...,...,...,...,...,...,...,...
4790,1000199,estavel,2483.11,13,3,19,15,12,13.000000,10,2483.110000
4791,1000199,estavel,2483.11,13,3,19,16,13,13.000000,22,2483.110000
4792,1000199,estavel,2483.11,13,3,19,17,14,13.000000,8,2483.110000
4793,1000199,estavel,2483.11,13,3,19,18,15,13.000000,7,2483.110000


In [28]:
# ======================================
# 6. EXPLOSÃO TRANSACIONAL
# ======================================

sigma_valor = 0.10

df_pgtos = df_fornecedor_mes_ativo.loc[df_fornecedor_mes_ativo.index.repeat(df_fornecedor_mes_ativo['quantidade_real_pagamentos'])].reset_index(drop=True)

display(df_pgtos[(df_pgtos['id_fornecedor_raw'] == '1000000') & (df_pgtos['mes_pagamento'] == 13)])

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes
28,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
29,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
30,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
31,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
32,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
33,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
34,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
35,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
36,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948


In [29]:
# ======================================
# 7. MODELAGEM MONETÁRIA
# ======================================

proporcao_ajuste = 0.3
desconto_min = 0.01 
desconto_max = 0.05 

N = len(df_pgtos)
array =  np.random.uniform(0, 1, N) # [1, 2, 3, 4, 5]
#print(array)
flag = array < proporcao_ajuste
#print(flag)
df_pgtos["flag_ajuste"] = flag

#df_pgtos[(df_pgtos['id_fornecedor_raw'] == '1000000') & (df_pgtos['mes_relativo'] == 2)]

# for i in range(len(df_pgtos)): 
#     prob_ajuste = np.random.uniform(0, 1) 
#     if prob_ajuste < 0.3: 
#         df_pgtos.loc[i, "flag_ajuste"] = True 
#     else: 
#         df_pgtos.loc[i, "flag_ajuste"] = False 

# #display(df_pgtos[(df_pgtos['flag_ajuste'] == True)]) 

# display(df_pgtos)

df_pgtos["mu_valor"] = np.log(df_pgtos['valor_medio_mes']) - (sigma_valor**2)/2

df_pgtos["valor_bruto"] = np.random.lognormal(df_pgtos["mu_valor"], sigma_valor)

display(df_pgtos)



df_pgtos["desconto"] = 0.0
mask_disk = df_pgtos['flag_ajuste'] == True

N = len(mask_disk)
desconto_min = 0.01 
desconto_max = 0.05 
desconto = np.random.uniform(desconto_min, desconto_max, N)

df_pgtos.loc[ 
    mask_disk, 
    'desconto' 
    ] = desconto[mask_disk]

df_pgtos['desconto'] = df_pgtos['valor_bruto'] * df_pgtos['desconto']

df_pgtos['valor_liquido'] = df_pgtos['valor_bruto'] - df_pgtos['desconto']
# df_pgtos[(df_pgtos['flag_ajuste'] == True)]

# df_pgtos[(df_pgtos['flag_ajuste'] == False) & (df_pgtos['desconto'] > 0)]

df_pgtos["id_transacao_raw"] = [f'TR-{i:05d}' for i in range(1, len(df_pgtos) + 1)]

df_pgtos

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste,mu_valor,valor_bruto
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3966.097441
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4475.940332
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4101.466781
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4421.324444
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3718.020715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,1976.352737
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2192.120003
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,3005.351230
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2185.524770


,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste,mu_valor,valor_bruto,desconto,valor_liquido,id_transacao_raw
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3966.097441,0.000000,3966.097441,TR-00001
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4475.940332,0.000000,4475.940332,TR-00002
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4101.466781,0.000000,4101.466781,TR-00003
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4421.324444,0.000000,4421.324444,TR-00004
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3718.020715,0.000000,3718.020715,TR-00005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,1976.352737,0.000000,1976.352737,TR-39788
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2192.120003,0.000000,2192.120003,TR-39789
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,3005.351230,0.000000,3005.351230,TR-39790
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2185.524770,0.000000,2185.524770,TR-39791


In [30]:
# ======================================
# 8. MODELAGEM TEMPORAL TRANSACIONAL
# ======================================

data_base = pd.to_datetime('2024-01-01')

df_pgtos['data'] = data_base + df_pgtos['mes_pagamento'].apply(lambda x: pd.DateOffset(months=x-1))

df_pgtos['dias_no_mes'] = pd.DatetimeIndex(df_pgtos['data']).days_in_month

triangulo = np.random.triangular(left=0, mode=0.5, right=1, size=len(df_pgtos))
#print(triangulo)

df_pgtos['dia_pgto'] = triangulo * (df_pgtos['dias_no_mes'])

# df_pgtos['triangulo'] = df_pgtos['triangulo'] 

df_pgtos['dia_pgto'] = df_pgtos.dia_pgto.astype(int)

df_pgtos['data_transacao'] = df_pgtos['data'] + pd.to_timedelta(df_pgtos['dia_pgto'] - 1, unit='D')

mask_disk = df_pgtos['flag_ajuste'] == True

df_pgtos['data_transacao'] = pd.to_datetime(
    df_pgtos['data_transacao'],
    errors='coerce'
)

# mask_domingo = df_pgtos['data'] + pd.to_timedelta(df_pgtos['dia_pgto'] - 1, unit='D') == 6
mask_domingo = df_pgtos['data_transacao'].dt.dayofweek == 6
mask_sabado = df_pgtos['data_transacao'].dt.dayofweek == 5

#domingo
df_pgtos.loc[
    mask_domingo,
    'data_transacao'
] = df_pgtos.loc[mask_domingo, 'data_transacao'] + pd.Timedelta(days=1)

#sabado
df_pgtos.loc[
    mask_sabado,
    'data_transacao'
] = df_pgtos.loc[mask_sabado, 'data_transacao'] + pd.Timedelta(days=2)

df_pgtos['data_competencia'] = df_pgtos['data_transacao'].dt.to_period('M')

df_pgtos[df_pgtos['valor_liquido'] == 0]


C:\Users\nasse\AppData\Local\Temp\ipykernel_10360\2795241379.py:20: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_pgtos['data_transacao'] = df_pgtos['data'] + pd.to_timedelta(df_pgtos['dia_pgto'] - 1, unit='D')


,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,mu_valor,valor_bruto,desconto,valor_liquido,id_transacao_raw,data,dias_no_mes,dia_pgto,data_transacao,data_competencia


In [31]:
# ======================================
# 9. SEGMENTAÇÃO DE PERFIL DE RISCO
# ======================================

P90 = df_pgtos['valor_bruto'].quantile(0.9)
df_pgtos['flag_alto_valor'] = df_pgtos['valor_bruto'] > P90

df_pgtos

# df_pgtos[df_pgtos['P90'] == True]

# perfil_risco

condicoes = [
    (df_pgtos['flag_ajuste'] == True) & (df_pgtos['flag_alto_valor'] == True) ,
    (df_pgtos['flag_ajuste'] == True) & (df_pgtos['flag_alto_valor'] == False),
    (df_pgtos['flag_ajuste'] == False) & (df_pgtos['flag_alto_valor'] == True),
    (df_pgtos['flag_ajuste'] == False) & (df_pgtos['flag_alto_valor'] == False)]

escolhas = ['alto_valor_retencao', 'retencao', 'alto_valor', 'normal']

df_pgtos['perfil_risco'] = np.select(condicoes, escolhas, default='Desconecido')

alto_valor = df_pgtos[df_pgtos['perfil_risco'] == 'alto_valor_retencao']['perfil_risco'].count()

In [32]:
# ======================================
# 10. VALIDAÇÕES ESTATÍSTICAS
# ======================================

display(df_pgtos)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,desconto,valor_liquido,id_transacao_raw,data,dias_no_mes,dia_pgto,data_transacao,data_competencia,flag_alto_valor,perfil_risco
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,3966.097441,TR-00001,2024-10-01 00:00:00,31,14,2024-10-14,2024-10,False,normal
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,4475.940332,TR-00002,2024-10-01 00:00:00,31,9,2024-10-09,2024-10,False,normal
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,4101.466781,TR-00003,2024-10-01 00:00:00,31,9,2024-10-09,2024-10,False,normal
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,4421.324444,TR-00004,2024-10-01 00:00:00,31,14,2024-10-14,2024-10,False,normal
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,0.000000,3718.020715,TR-00005,2024-10-01 00:00:00,31,10,2024-10-10,2024-10,False,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,1976.352737,TR-39788,2025-07-01 00:00:00,31,22,2025-07-22,2025-07,False,normal
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,2192.120003,TR-39789,2025-07-01 00:00:00,31,21,2025-07-21,2025-07,False,normal
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,3005.351230,TR-39790,2025-07-01 00:00:00,31,18,2025-07-18,2025-07,False,normal
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,0.000000,2185.524770,TR-39791,2025-07-01 00:00:00,31,26,2025-07-28,2025-07,False,normal


In [33]:
# ======================================
# 11. DEFINIÇÃO DO TIPO DE TRANSAÇÃO
# ======================================

# for i in range(len(df_pgtos)):
#     tipo_transacao = np.random.uniform(0, 1)
#     df_pgtos.loc[i, 'tipo_transacao'] = tipo_transacao

N = len(df_pgtos)
array = np.random.uniform(0, 1, N)
flag_tipo = array < 0.6 

# Criar coluna tipo_transacao ao invés de sobrescrever flag_ajuste
df_pgtos['tipo_transacao'] = np.where(flag_tipo, 'DESPESA', 'RECEITA')

df_pgtos['status_pagamento'] = None

mask_despesa_normal = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'normal')
mask_despesa_retencao = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'retencao')
mask_despesa_alto_valor = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'alto_valor')
mask_despesa_alto_valor_retencao = (df_pgtos['tipo_transacao'] == 'DESPESA') & (df_pgtos['perfil_risco'] == 'alto_valor_retencao')

#print(mask_despesa)

mask_receita_normal = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'normal')
mask_receita_retencao = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'retencao')
mask_receita_alto_valor = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'alto_valor')
mask_receita_alto_valor_retencao = (df_pgtos['tipo_transacao'] == 'RECEITA') & (df_pgtos['perfil_risco'] == 'alto_valor_retencao')

def aplicar_status(mask, p_pago, p_atrasado, p_cancelado):
    N = mask.sum()

    if N == 0:
        return  # sair da função

    sorteios = np.random.uniform(0, 1, N)

    # 3. Definir limites acumulados
    limite_pago = p_pago
    limite_atrasado = p_pago + p_atrasado
    # o restante será cancelado

    status_grupo = np.empty(N, dtype=object)

    for i, sorteio in enumerate(sorteios):
        if sorteio < limite_pago:
            status_grupo[i] = "PAGO"
        elif sorteio < limite_atrasado:
            status_grupo[i] = "ATRASADO"
        else:
            status_grupo[i] = "CANCELADO"

    # 5. Atribuir esse array apenas nas linhas da máscara
    df_pgtos.loc[mask, 'status_pagamento'] = status_grupo

aplicar_status(mask_despesa_normal, 0.85, 0.10, 0.05)
aplicar_status(mask_despesa_retencao, 0.72, 0.20, 0.08)
aplicar_status(mask_despesa_alto_valor, 0.92, 0.06, 0.02)
aplicar_status(mask_despesa_alto_valor_retencao, 0.80, 0.14, 0.06)
aplicar_status(mask_receita_normal, 0.80, 0.14, 0.06)
aplicar_status(mask_receita_retencao, 0.68, 0.24, 0.08)
aplicar_status(mask_receita_alto_valor, 0.88, 0.09, 0.03)
aplicar_status(mask_receita_alto_valor_retencao, 0.75, 0.17, 0.08)

df_pgtos['status_pagamento'].isnull().sum()

df_pgtos

df_base_normal = df_pgtos.copy()

In [34]:
# 12. INJEÇÃO DE ANOMALIA DE VALOR

anomalias_por_fornecedor = []

df_contagem_fornecedores = df_pgtos.groupby('id_fornecedor_raw').size().reset_index(name='qtd_transacoes')

df_contagem_fornecedores

for fornecedor in df_contagem_fornecedores['id_fornecedor_raw']:
    
    subset_fornecedor = df_pgtos[df_pgtos['id_fornecedor_raw'] == fornecedor]

    qtd_transacoes = len(subset_fornecedor)

    qtd_anomalias = int(np.ceil(qtd_transacoes * tx_anomalia_mes))

    if qtd_anomalias == 0:
        continue

    anomalias_por_fornecedor.append({
        'id_fornecedor_raw': fornecedor,
        'qtd_transacoes': qtd_transacoes,
        'qtd_anomalias': qtd_anomalias,
        'percentual': (qtd_anomalias / qtd_transacoes) * 100
    })

df_anomalias_config = pd.DataFrame(anomalias_por_fornecedor)

df_anomalias_config

,id_fornecedor_raw,qtd_transacoes,qtd_anomalias,percentual
0,1000000,139,9,6.474820
1,1000004,198,12,6.060606
2,1000014,147,9,6.122449
3,1000015,123,8,6.504065
4,1000016,147,9,6.122449
...,...,...,...,...
195,FORN-00192,111,7,6.306306
196,FORN-00193,162,10,6.172840
197,FORN-00195,427,26,6.088993
198,FORN-00196,173,11,6.358382


In [35]:
for_ids = []

map_valor = df_pgtos.set_index('id_transacao_raw')['valor_bruto'].to_dict()

for fornecedor in df_anomalias_config['id_fornecedor_raw']:
    subset_fornecedor = df_anomalias_config[df_anomalias_config['id_fornecedor_raw'] == fornecedor]
    
    fornecedor = subset_fornecedor['id_fornecedor_raw'].values[0]
    qtd_anomalias = int(subset_fornecedor['qtd_anomalias'].values[0])
    id_transacao_raw = df_pgtos[df_pgtos['id_fornecedor_raw'] == fornecedor]['id_transacao_raw'].tolist()

    qtd_anomalias = min(qtd_anomalias, len(id_transacao_raw))

    if qtd_anomalias <= 0 or len(id_transacao_raw) == 0:
        continue

    ids = np.random.choice(id_transacao_raw, size=qtd_anomalias, replace=False)
    
    ids_lista = ids.tolist()

    for id_sorteado in ids:
        if np.random.uniform(0, 1) < 0.7:
            fator_aplicado = np.random.uniform(1.5, 3.5)  # Aumento entre 50% e 250%
        else: 
            fator_aplicado = np.random.uniform(0.2, 0.6)  # Redução entre 20% e 60%

        for_ids.append({
            'id_fornecedor_raw': fornecedor,
            'qtd_anomalias': qtd_anomalias,
            'id_transacao_raw': id_sorteado, 
            'fator_aplicado': float(fator_aplicado),
            'valor_original': map_valor[id_sorteado],
            'valor_novo': map_valor[id_sorteado] * fator_aplicado
        })

df_anomalias_valor = pd.DataFrame(for_ids)

In [48]:
df_base_anomalo = df_base_normal.copy()

df_base_anomalo['flag_anomalia'] = 0

# merge anomalies; avoid duplicate fornecedor columns

df_base_anomalo = df_base_anomalo.merge(
    df_anomalias_valor,
    on='id_transacao_raw',
    how='left'
)

# after merge pandas appends _x/_y suffixes for shared column names
# keep the original left-side id_fornecedor_raw and drop any extra
if 'id_fornecedor_raw_y' in df_base_anomalo.columns:
    df_base_anomalo.drop(columns=['id_fornecedor_raw_y'], inplace=True)
if 'id_fornecedor_raw_x' in df_base_anomalo.columns:
    df_base_anomalo.rename(columns={'id_fornecedor_raw_x': 'id_fornecedor_raw'}, inplace=True)

# flag anomalies based on factor

df_base_anomalo['flag_anomalia'] = df_base_anomalo['fator_aplicado'] >= 0

mask_anomalo = df_base_anomalo['flag_anomalia'] == True

df_base_anomalo.loc[
    mask_anomalo,
    'valor_bruto'
] = df_base_anomalo.loc[mask_anomalo, 'valor_novo']

df_base_anomalo['valor_liquido'] = df_base_anomalo['valor_bruto'] - df_base_anomalo['desconto']

# teste = df_base_anomalo[['valor_bruto', 'id_transacao_raw', 'flag_anomalia']]

# teste[teste['flag_anomalia'] == True]

# df_base_anomalo[df_base_anomalo['id_transacao_raw'] == 'TR-00022']



In [37]:
# áreas

id_areas = ['FIN', 'COMP', 'OP', 'COM', 'RH', 'TI', 'MKT', 'JUR']
peso = [0.15, 0.18, 0.17, 0.14, 0.10, 0.09, 0.09, 0.08]

areas = ['Financeiro', 'Compras', 'Operações', 'Comercial', 'RH', 'TI', 'Marketing', 'Jurídico']

N = len(df_base_anomalo)

array_areas = np.random.choice(id_areas, p=peso, size=N)

df_base_anomalo["id_area_raw"] = array_areas

df_base_anomalo

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,flag_alto_valor,perfil_risco,tipo_transacao,status_pagamento,flag_anomalia,qtd_anomalias,fator_aplicado,valor_original,valor_novo,id_area_raw
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,normal,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,COM
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,OP
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,JUR
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,COM
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,normal,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,COM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,normal,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,JUR
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,TI
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,TI
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,COM


In [38]:
# categoria

Categoria = [
	'Serviços',
	'Material',
	'Infraestrutura',
	'Marketing',
	'Tecnologia',
	'Recursos Humanos',
	'Jurídico',
	'Operacional']

id_categoria = [1,2,3,4,5,6,7,8]

peso_categoria = [0.22, 0.18, 0.11, 0.08, 0.12, 0.07, 0.05, 0.17]

array_categoria = np.random.choice(id_categoria, p=peso_categoria, size=N)

df_base_anomalo["id_categoria_raw"] = array_categoria

df_base_anomalo

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,perfil_risco,tipo_transacao,status_pagamento,flag_anomalia,qtd_anomalias,fator_aplicado,valor_original,valor_novo,id_area_raw,id_categoria_raw
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,normal,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,COM,7
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,OP,5
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,JUR,5
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,COM,2
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,normal,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,COM,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,normal,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,JUR,8
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,TI,2
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,TI,5
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,normal,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,COM,1


In [39]:
# moeda

moeda = [
	'BRL',
    'USD',
    'EUR']

peso_moeda = [0.94, 0.05, 0.01]

array_moeda = np.random.choice(moeda, p=peso_moeda, size=N)

df_base_anomalo["moeda"] = array_moeda

df_base_anomalo

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,tipo_transacao,status_pagamento,flag_anomalia,qtd_anomalias,fator_aplicado,valor_original,valor_novo,id_area_raw,id_categoria_raw,moeda
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,COM,7,BRL
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,OP,5,BRL
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,JUR,5,BRL
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,COM,2,BRL
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,COM,1,BRL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,RECEITA,PAGO,False,NaN,NaN,NaN,NaN,JUR,8,BRL
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,TI,2,BRL
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,TI,5,BRL
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,DESPESA,PAGO,False,NaN,NaN,NaN,NaN,COM,1,BRL


In [40]:
# forma de pagamento

forma_pagamento = ['PIX', 'BOLETO', 'TED', 'TRANSFERENCIA', 'CARTAO']

peso_forma_pagamento = [0.28, 0.24, 0.20, 0.18, 0.10]

escolha_tp_pgto = np.random.choice(forma_pagamento, p=peso_forma_pagamento, size=N)

df_base_anomalo["forma_pagamento"] = escolha_tp_pgto

df_base_anomalo

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,status_pagamento,flag_anomalia,qtd_anomalias,fator_aplicado,valor_original,valor_novo,id_area_raw,id_categoria_raw,moeda,forma_pagamento
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,COM,7,BRL,BOLETO
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,OP,5,BRL,TED
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,JUR,5,BRL,PIX
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,COM,2,BRL,PIX
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,PAGO,False,NaN,NaN,NaN,NaN,COM,1,BRL,BOLETO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,JUR,8,BRL,PIX
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,TI,2,BRL,PIX
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,TI,5,BRL,TRANSFERENCIA
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,PAGO,False,NaN,NaN,NaN,NaN,COM,1,BRL,TED


In [41]:
# Mapear id_categoria_raw para nome
map_categoria = {
    1: "Serviços",
    2: "Material",
    3: "Infraestrutura",
    4: "Marketing",
    5: "Tecnologia",
    6: "Recursos Humanos",
    7: "Jurídico",
    8: "Operacional"
}

# Templates por categoria
templates = {
    1: [
        "Prestação de serviço para área {area}",
        "Contrato de serviço - setor {area}",
        "Serviço especializado solicitado por {area}"
    ],
    2: [
        "Aquisição de material operacional - {area}",
        "Compra de insumos para setor {area}"
    ],
    3: [
        "Despesa de infraestrutura - {area}",
        "Manutenção estrutural vinculada à área {area}"
    ],
    4: [
        "Campanha institucional - {area}",
        "Investimento em marketing corporativo ({area})"
    ],
    5: [
        "Licenciamento de software - {area}",
        "Serviço de tecnologia contratado por {area}"
    ],
    6: [
        "Despesa de RH - {area}",
        "Serviço relacionado a recursos humanos ({area})"
    ],
    7: [
        "Honorários jurídicos - {area}",
        "Assessoria legal vinculada à área {area}"
    ],
    8: [
        "Despesa operacional - {area}",
        "Custo operacional associado a {area}"
    ]
}

def gerar_descricao(row):
    categoria = row["id_categoria_raw"]
    area = row["id_area_raw"]
    template = np.random.choice(templates[categoria])
    return template.format(area=area)

df_base_anomalo["descricao"] = df_base_anomalo.apply(gerar_descricao, axis=1)

df_base_anomalo

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,flag_anomalia,qtd_anomalias,fator_aplicado,valor_original,valor_novo,id_area_raw,id_categoria_raw,moeda,forma_pagamento,descricao
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,NaN,NaN,NaN,NaN,COM,7,BRL,BOLETO,Assessoria legal vinculada à área COM
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,NaN,NaN,NaN,NaN,OP,5,BRL,TED,Serviço de tecnologia contratado por OP
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,NaN,NaN,NaN,NaN,JUR,5,BRL,PIX,Licenciamento de software - JUR
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,NaN,NaN,NaN,NaN,COM,2,BRL,PIX,Aquisição de material operacional - COM
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,NaN,NaN,NaN,NaN,COM,1,BRL,BOLETO,Serviço especializado solicitado por COM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,NaN,NaN,NaN,NaN,JUR,8,BRL,PIX,Despesa operacional - JUR
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,NaN,NaN,NaN,NaN,TI,2,BRL,PIX,Compra de insumos para setor TI
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,NaN,NaN,NaN,NaN,TI,5,BRL,TRANSFERENCIA,Licenciamento de software - TI
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,NaN,NaN,NaN,NaN,COM,1,BRL,TED,Contrato de serviço - setor COM


In [42]:
import uuid

df_base_anomalo["data_competencia"] = df_base_anomalo["data_competencia"].dt.to_timestamp()

df_lotes = (
    df_base_anomalo[["data_competencia"]]
    .drop_duplicates()
    .sort_values("data_competencia")
    .reset_index(drop=True)
)

df_lotes["ingestion_id"] = [str(uuid.uuid4()) for _ in range(len(df_lotes))]

df_lotes["ingestion_ts"] = (
    df_lotes["data_competencia"] + pd.DateOffset(months=1) + pd.DateOffset(days=4)
)

df_base_anomalo = df_base_anomalo.merge(
    df_lotes,
    on="data_competencia",
    how="left"
)

df_base_anomalo["source_system"] = "ERP_CORPORATIVO"
df_base_anomalo["source_entity"] = "TRANSACOES_FINANCEIRAS"
df_base_anomalo["row_seq"] = range(1, len(df_base_anomalo) + 1)

In [51]:
import hashlib

colunas_hash = [
    "id_transacao_raw",
    "data_transacao",
    "data_competencia",
    "id_fornecedor_raw",
    "id_area_raw",
    "id_categoria_raw",
    "tipo_transacao",
    "valor_bruto",
    "valor_liquido",
    "status_pagamento",
    "moeda",
    "forma_pagamento",
    "descricao"
]

for col in colunas_hash:
    if col not in df_base_anomalo.columns:
        df_base_anomalo[col] = None

print("Colunas antes do hash:", df_base_anomalo.columns.tolist())

def gerar_hash_linha(row):
    valores = []
    for col in colunas_hash:
        valor = row.get(col, "") 
        if pd.isnull(valor):
            valor = ""
        valores.append(str(valor))
    texto_concat = "|".join(valores)
    return hashlib.sha256(texto_concat.encode("utf-8")).hexdigest()

df_base_anomalo["raw_row_hash"] = df_base_anomalo.apply(gerar_hash_linha, axis=1)

Colunas antes do hash: ['id_fornecedor_raw', 'grupo', 'valor_medio_inicial', 'freq_media_inicial', 'mes_inicio', 'mes_fim', 'mes_pagamento', 'mes_relativo', 'lambda_mes', 'quantidade_real_pagamentos', 'valor_medio_mes', 'flag_ajuste', 'mu_valor', 'valor_bruto', 'desconto', 'valor_liquido', 'id_transacao_raw', 'data', 'dias_no_mes', 'dia_pgto', 'data_transacao', 'data_competencia', 'flag_alto_valor', 'perfil_risco', 'tipo_transacao', 'status_pagamento', 'flag_anomalia', 'qtd_anomalias', 'fator_aplicado', 'valor_original', 'valor_novo', 'id_area_raw', 'id_categoria_raw', 'moeda', 'forma_pagamento', 'descricao']


In [44]:
colunas_raw = [
    "id_transacao_raw",
    "data_transacao",
    "data_competencia",
    "id_area_raw",
    "id_fornecedor_raw",
    "tipo_transacao",
    "valor_bruto",
    "valor_liquido",
    "moeda",
    "descricao",
    "id_categoria_raw",
    "forma_pagamento",
    "status_pagamento",
    "ingestion_id",
    "ingestion_ts",
    "source_system",
    "source_entity",
    "row_seq",
    "raw_row_hash"
]

df_pgtos = df_base_anomalo[colunas_raw].copy()

In [45]:
df_pgtos_bkp = df_pgtos.copy()

df_pgtos_bkp["flag_anomalia"] = df_base_anomalo["flag_anomalia"]

In [54]:
import os

# compute path relative to workspace root instead of notebook directory
cwd = os.getcwd()
workspace_root = os.path.abspath(os.path.join(cwd, "..", ".."))
pasta_destino = os.path.join(workspace_root, "data", "raw", "transacoes_financeiras")
os.makedirs(pasta_destino, exist_ok=True)
print("Working directory:", cwd)
print("Export folder:", pasta_destino)

df_pgtos["data_competencia"] = df_pgtos["data_competencia"].astype(str)

# Gerar um CSV por competência
for competencia, df_mes in df_pgtos.groupby("data_competencia"):
    
    # Nome padrão do arquivo
    nome_comp = competencia.replace("-", "_")
    caminho_arquivo = os.path.join(
        pasta_destino,
        f"transacoes_{nome_comp}.csv"
    )
    
    # Exportar
    df_mes.to_csv(
        caminho_arquivo,
        index=False,
        sep=",",
        encoding="utf-8"
    )

print("Exportação concluída com sucesso.")

Working directory: d:\Faculdade\TCC\sql\dml
Export folder: d:\Faculdade\TCC\data\raw\transacoes_financeiras
Exportação concluída com sucesso.
